# Example: Recurrent Neural Networks for Time Series Prediction
In this example, we apply an Elman RNN to forecast thrombin generation curves from the tissue factor-initiated blood coagulation cascade. We generate training data by simulating the [Hockin-Mann coagulation model](https://github.com/varnerlab/HockinMannModel.jl) at different tissue factor (TF) concentrations, train an Elman RNN to predict the full thrombin time course given a TF level, and evaluate its predictions on unseen TF concentrations.

> __Learning Objectives:__
>
> By the end of this example, you should be able to:
>
> * __Generate thrombin time series data from ODE simulations:__ Use the Hockin-Mann coagulation model to produce thrombin generation curves at different tissue factor concentrations and organize the data for RNN training.
> * __Implement and train an Elman RNN for autoregressive prediction:__ Build an Elman RNN from scratch, train it using teacher forcing with gradient-based optimization, and connect the architecture to the equations from the lecture.
> * __Evaluate RNN predictions on unseen conditions:__ Use autoregressive rollout to generate full predicted curves on held-out TF concentrations and compare predictions to ground truth using thrombin generation assay features.

Let's get started!
___

## Setup, Data, and Prerequisites
We set up the computational environment by including the `Include.jl` file, loading any needed resources, such as sample datasets, and setting up any required constants.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, and includes local source files in `src/`. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

In [1]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

In addition to standard Julia libraries, this example uses [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl) and [the `HockinMannModel.jl` package](https://github.com/varnerlab/HockinMannModel.jl). The `HockinMannModel.jl` package provides ODE-based simulations of the tissue factor-initiated coagulation cascade, including functions to compute thrombin generation curves and extract thrombin generation assay (TGA) features such as peak thrombin, time-to-peak, and lag time.

### Implementations
We use helper functions defined in the `src/` directory:

| Function | Source | Description |
| --- | --- | --- |
| [`generate_thrombin_dataset(...)`](src/Simulation.jl) | `src/Simulation.jl` | Simulate the Hockin-Mann model at multiple TF levels and return thrombin curves (nM) |
| [`normalize_minmax(...)`](src/Simulation.jl) | `src/Simulation.jl` | Min-max normalize a data matrix to $[0,1]$ |
| [`denormalize_minmax(...)`](src/Simulation.jl) | `src/Simulation.jl` | Reverse min-max normalization to recover original scale |
| [`build_elman_rnn(...)`](src/ElmanRNN.jl) | `src/ElmanRNN.jl` | Initialize an Elman RNN with Xavier-scaled weights |
| [`forward_step(...)`](src/ElmanRNN.jl) | `src/ElmanRNN.jl` | Compute one time step of the Elman RNN |
| [`forward_sequence(...)`](src/ElmanRNN.jl) | `src/ElmanRNN.jl` | Run the RNN over a full sequence with teacher forcing |
| [`predict_sequence(...)`](src/ElmanRNN.jl) | `src/ElmanRNN.jl` | Generate a predicted sequence via autoregressive rollout |
| [`count_parameters(...)`](src/ElmanRNN.jl) | `src/ElmanRNN.jl` | Count total trainable parameters |

### Constants
We define simulation and training constants used throughout the example. The simulation constants control the TF concentration range and time resolution, while the RNN constants specify the network architecture and training hyperparameters.

In [1]:
# simulation constants -
number_of_curves = 50;          # total number of thrombin curves to generate
tf_min = 1e-12;                 # minimum TF concentration: 1 pM
tf_max = 50e-12;                # maximum TF concentration: 50 pM
saveat = 10.0;                  # save ODE solution every 10 seconds (121 time points)
tspan = (0.0, 1200.0);         # simulation time span: 0 to 1200 seconds (20 minutes)

# RNN architecture constants -
d_in = 2;                       # input dimension: [x_t, TF_normalized]
h_dim = 64;                     # hidden state dimension
d_out = 1;                      # output dimension: predicted x_{t+1}

# training constants -
num_epochs = 500;               # number of training epochs
learning_rate = 1e-3;           # Adam optimizer learning rate
train_fraction = 0.8;           # fraction of curves used for training (80% train, 20% test)
grad_clip = 1.0;                # gradient clipping threshold (max gradient norm)

___

## Task 1: Generate Thrombin Generation Data
We use the Hockin-Mann 2002 coagulation model to generate thrombin generation curves at different tissue factor (TF) concentrations. TF initiates the extrinsic coagulation pathway, and its concentration controls the overall dynamics of thrombin generation.

> __Tissue factor titration__
>
> Tissue factor (TF) is a transmembrane glycoprotein that initiates the coagulation cascade. In a thrombin generation assay (TGA), increasing TF concentration shortens the lag time, increases the peak thrombin concentration, and shifts the peak to earlier time points. We simulate 50 curves at TF concentrations logarithmically spaced from 1 pM to 50 pM using [the `generate_thrombin_dataset(...)` function](src/Simulation.jl), which internally calls [the `simulate(...)` function](https://github.com/varnerlab/HockinMannModel.jl/blob/main/src/Simulate.jl) and [the `total_thrombin(...)` function](https://github.com/varnerlab/HockinMannModel.jl/blob/main/src/Simulate.jl) from the `HockinMannModel.jl` package.

The TF concentrations are generated using the [`range(...)` function](https://docs.julialang.org/en/v1/base/math/#Base.range) in log-space. The simulation results are stored in the `time_vector::Vector{Float64}` variable (time points in seconds), the `thrombin_matrix::Matrix{Float64}` variable (rows are time points, columns are different TF concentrations, values in nM), and the `tf_values::Vector{Float64}` variable (TF concentrations in molar).

In [2]:
time_vector, thrombin_matrix, tf_values = let

    # generate TF concentrations (log-spaced from 1 pM to 50 pM) -
    tf_values = 10 .^ range(log10(tf_min), log10(tf_max), length=number_of_curves);

    # simulate the Hockin-Mann model at each TF level, returns (time, thrombin_nM, tf) -
    time_vector, thrombin_matrix, tf_values = generate_thrombin_dataset(tf_values;
        tspan=tspan, saveat=saveat);

    # visualize: plot all thrombin curves colored by TF concentration -
    p = plot(xlabel="Time (s)", ylabel="Total Thrombin (nM)", legend=:topright);
    colors = palette(:viridis, number_of_curves); # color gradient from low to high TF
    for j in 1:number_of_curves
        # only label the first and last curves to keep the legend clean -
        label_str = (j == 1 || j == number_of_curves) ? "TF = $(round(tf_values[j]*1e12, digits=1)) pM" : "";
        plot!(p, time_vector, thrombin_matrix[:, j], label=label_str, color=colors[j], lw=1.5);
    end
    p

    # return simulation data -
    (time_vector, thrombin_matrix, tf_values)
end;

UndefVarError: UndefVarError: `generate_thrombin_dataset` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

The curves show that higher TF concentrations produce earlier and larger thrombin peaks, while lower TF concentrations produce delayed and smaller peaks.

Next, we normalize the thrombin data using [the `normalize_minmax(...)` function](src/Simulation.jl) and split into training and test sets using [the `randperm(...)` function](https://docs.julialang.org/en/v1/stdlib/Random/#Random.randperm). The normalized thrombin curves are stored in `X_norm::Matrix{Float64}` with values in $[0,1]$, the normalization bounds in `thrombin_min::Float64` and `thrombin_max::Float64`, the normalized TF values in `tf_norm::Vector{Float64}`, and the training and test indices in `train_idx::Vector{Int}` and `test_idx::Vector{Int}`.

In [3]:
X_norm, thrombin_min, thrombin_max, tf_norm, tf_norm_min, tf_norm_max, train_idx, test_idx = let

    # normalize thrombin curves to [0, 1] using min-max scaling -
    X_norm, thrombin_min, thrombin_max = normalize_minmax(thrombin_matrix);

    # normalize TF values to [0, 1] using log-transform then min-max scaling -
    # log-transform because TF spans orders of magnitude (1 pM to 50 pM) -
    tf_log = log10.(tf_values);
    tf_norm_min = minimum(tf_log);
    tf_norm_max = maximum(tf_log);
    tf_norm = (tf_log .- tf_norm_min) ./ (tf_norm_max - tf_norm_min);

    # shuffle curve indices and split into train/test sets -
    Random.seed!(42); # set seed for reproducibility
    perm = randperm(number_of_curves); # random permutation of curve indices
    n_train = Int(round(train_fraction * number_of_curves)); # number of training curves
    train_idx = perm[1:n_train];           # first 80% for training
    test_idx = perm[(n_train+1):end];      # remaining 20% for testing

    # print dataset summary -
    println("Dataset: $(number_of_curves) curves, $(size(X_norm, 1)) time points each");
    println("Training: $(length(train_idx)) curves, Test: $(length(test_idx)) curves");
    println("Thrombin range: $(round(thrombin_min, digits=2)) to $(round(thrombin_max, digits=2)) nM");

    # return all outputs -
    (X_norm, thrombin_min, thrombin_max, tf_norm, tf_norm_min, tf_norm_max, train_idx, test_idx)
end;

UndefVarError: UndefVarError: `normalize_minmax` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___

## Task 2: Build the Elman RNN
We construct an Elman RNN following the architecture from the [L12a lecture on Recurrent Neural Networks](CHEME-5820-L12a-Lecture-RecurrentNetworks-Spring-2026.ipynb). The RNN operates in an autoregressive mode: at each time step $t$, it receives the current thrombin value $x_t$ concatenated with the conditioning parameter (normalized TF concentration) and predicts the next value $x_{t+1}$.

> __Autoregressive Elman RNN__
>
> The Elman RNN equations from the lecture are:
> $$
\begin{align*}
\mathbf{h}_t &= \tanh(\mathbf{U}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b}_h) \\
\mathbf{y}_t &= \mathbf{W}_y \mathbf{h}_t + \mathbf{b}_y
\end{align*}
> $$
> For autoregressive time series prediction, we set $\mathbf{x}_t = [x_t, c]^\top \in \mathbb{R}^{2}$ where $x_t$ is the normalized thrombin value at time $t$ and $c$ is the normalized TF concentration. The output $y_t \in \mathbb{R}$ is the predicted value $\hat{x}_{t+1}$. The output uses a linear activation (no nonlinearity) because this is a regression task.

We use [the `build_elman_rnn(...)` function](src/ElmanRNN.jl) to initialize the model with Xavier-scaled random weights and zero biases. The model is stored in the `model::MyElmanRNNModel` variable, which holds the weight matrices $\mathbf{U}_h$, $\mathbf{W}_x$, $\mathbf{W}_y$ and bias vectors $\mathbf{b}_h$, $\mathbf{b}_y$. We verify the parameter count using [the `count_parameters(...)` function](src/ElmanRNN.jl).

In [4]:
model = let

    # build the Elman RNN with d_in=2, h=64, d_out=1 -
    model = build_elman_rnn(d_in, h_dim, d_out; seed=42);

    # verify parameter count matches the lecture formula: h(h + d_in + 1) + d_out(h + 1) -
    expected = h_dim * (h_dim + d_in + 1) + d_out * (h_dim + 1);
    actual = count_parameters(model);
    println("Elman RNN: d_in=$(d_in), h=$(h_dim), d_out=$(d_out)");
    println("Expected parameters: $(expected), Actual: $(actual)");

    # return initialized model -
    model
end;

UndefVarError: UndefVarError: `build_elman_rnn` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___

## Task 3: Train the RNN
We train the Elman RNN using teacher forcing: at each time step during training, the model receives the true previous value $x_t$ (not its own prediction) and learns to predict $x_{t+1}$. The loss is the mean squared error between predicted and true values, averaged over all time steps and training curves.

> __Teacher forcing and gradient computation__
>
> During training, teacher forcing provides the ground truth $x_t$ at each step, which stabilizes learning by preventing error accumulation. We use [the `Flux.withgradient(...)` function](https://fluxml.ai/Flux.jl/stable/training/training/#Flux.withgradient-Tuple{Any,%20Vararg{Any}}) to compute gradients of the loss with respect to all model parameters, while keeping the explicit Elman RNN forward pass defined in [the `forward_sequence(...)` function](src/ElmanRNN.jl). We combine [`ClipGrad`](https://fluxml.ai/Optimisers.jl/dev/api/#Optimisers.ClipGrad) with the [`Adam`](https://fluxml.ai/Flux.jl/stable/training/optimisers/#Flux.Optimise.Adam) optimizer using [`OptimiserChain`](https://fluxml.ai/Optimisers.jl/dev/api/#Optimisers.OptimiserChain) to clip gradients before each update. Gradient clipping prevents the exploding gradient problem discussed in the lecture, which is common when backpropagating through many time steps.

The per-epoch average MSE loss values are stored in the `loss_history::Vector{Float64}` variable.

In [5]:
loss_history = let

    # initialize optimizer with gradient clipping to prevent exploding gradients -
    # OptimiserChain applies ClipGrad first, then Adam update -
    opt_state = Flux.setup(OptimiserChain(ClipGrad(grad_clip), Adam(learning_rate)), model);
    history = Vector{Float64}();   # stores average loss per epoch
    T = size(X_norm, 1);           # number of time points per curve

    # training loop: iterate over epochs -
    for epoch in 1:num_epochs
        epoch_loss = 0.0; # accumulate loss across all training curves

        # iterate over each training curve -
        for idx in train_idx

            # prepare this curve's data -
            curve = reshape(X_norm[:, idx], :, 1); # reshape to T x 1 matrix for forward_sequence
            condition = tf_norm[idx];                # normalized TF concentration for this curve
            targets = curve[2:end, 1];               # true x_{t+1} values (teacher forcing targets)

            # compute MSE loss and gradients via Flux.withgradient -
            loss_val, grads = Flux.withgradient(model) do m
                preds, _ = forward_sequence(m, curve, condition); # teacher forcing forward pass
                sum((preds[:, 1] .- targets) .^ 2) / (T - 1)    # mean squared error over time
            end

            # apply gradient update to model parameters (clipped then Adam) -
            Flux.update!(opt_state, model, grads[1]);
            epoch_loss += loss_val; # accumulate loss for this epoch
        end

        # record average loss for this epoch -
        push!(history, epoch_loss / length(train_idx));

        # print progress every 100 epochs -
        if epoch % 100 == 0 || epoch == 1
            println("Epoch $(epoch)/$(num_epochs): loss = $(round(history[end], sigdigits=4))");
        end
    end

    # return the loss history -
    history
end;

UndefVarError: UndefVarError: `Flux` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Let's plot the training loss to verify convergence.

In [6]:
let
    # plot loss history on log scale to visualize convergence -
    plot(1:num_epochs, loss_history, xlabel="Epoch", ylabel="Average MSE Loss",
        label="Training Loss", lw=2, yscale=:log10)
end

UndefVarError: UndefVarError: `plot` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___

## Task 4: Evaluate Predictions
We evaluate the trained RNN on the held-out test curves using autoregressive rollout. Starting from the true initial value $x_0$, the model feeds its own predictions back as input at each subsequent time step.

> __Autoregressive rollout vs. teacher forcing__
>
> During training, the model sees the true previous value at each step (teacher forcing). During evaluation, the model must use its own predictions, which means small errors can compound over time. This autoregressive evaluation is the true test of the model's ability to capture the dynamics of the system. [The `predict_sequence(...)` function](src/ElmanRNN.jl) implements this rollout, and [the `denormalize_minmax(...)` function](src/Simulation.jl) converts predictions back to nM.

Let's generate predictions for the test curves and compare them to the ground truth.

In [7]:
let

    # setup: determine how many test curves to plot -
    n_test = length(test_idx);
    T = size(X_norm, 1);                                       # number of time points
    n_plots = min(n_test, 4);                                   # plot up to 4 test curves
    sorted_test = sort(test_idx, by=i -> tf_values[i]);         # sort by TF concentration for display
    plot_indices = sorted_test[round.(Int, range(1, n_test, length=n_plots))]; # evenly spaced selection

    # generate predictions and create subplots -
    plots_array = [];
    for idx in plot_indices

        # autoregressive prediction: start from true x_0 and roll forward -
        x_0 = X_norm[1, idx];              # initial normalized thrombin value
        condition = tf_norm[idx];           # normalized TF concentration for this curve
        pred_norm = predict_sequence(model, x_0, condition, T); # generate T predicted steps

        # convert predictions back to nM using denormalize_minmax -
        pred_nM = denormalize_minmax(pred_norm, thrombin_min, thrombin_max);
        actual_nM = thrombin_matrix[:, idx]; # ground truth curve in nM
        tf_pM = round(tf_values[idx] * 1e12, digits=1); # TF in pM for display

        # plot actual vs predicted for this test curve -
        p = plot(time_vector, actual_nM, label="Actual", lw=2, color=:black);
        plot!(p, time_vector, pred_nM, label="Predicted", lw=2, color=:red, ls=:dash);
        plot!(p, xlabel="Time (s)", ylabel="Thrombin (nM)", title="TF = $(tf_pM) pM");
        push!(plots_array, p);
    end

    # combine into a 2x2 grid -
    plot(plots_array..., layout=(2, 2), size=(900, 600))
end

UndefVarError: UndefVarError: `test_idx` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

Let's quantify the prediction quality using TGA features (peak thrombin and time-to-peak) for each test curve. We compare actual and predicted values using [the `argmax(...)` function](https://docs.julialang.org/en/v1/base/collections/#Base.argmax) to find peak locations and [the `mean(...)` function](https://docs.julialang.org/en/v1/stdlib/Statistics/#Statistics.mean) to compute MSE.

In [8]:
let

    # initialize -
    T = size(X_norm, 1); # number of time points
    rows = [];           # collect table rows as named tuples

    # iterate over test curves sorted by TF concentration -
    for idx in sort(test_idx, by=i -> tf_values[i])

        # generate autoregressive prediction for this test curve -
        x_0 = X_norm[1, idx];              # initial normalized value
        condition = tf_norm[idx];           # normalized TF concentration
        pred_norm = predict_sequence(model, x_0, condition, T);

        # convert back to nM -
        pred_nM = denormalize_minmax(pred_norm, thrombin_min, thrombin_max);
        actual_nM = thrombin_matrix[:, idx];

        # compute TGA features: peak thrombin, time-to-peak, and MSE -
        actual_peak = maximum(actual_nM);                # actual peak thrombin (nM)
        actual_tpeak = time_vector[argmax(actual_nM)];   # actual time-to-peak (s)
        pred_peak = maximum(pred_nM);                    # predicted peak thrombin (nM)
        pred_tpeak = time_vector[argmax(pred_nM)];       # predicted time-to-peak (s)
        mse = mean((pred_nM .- actual_nM) .^ 2);        # mean squared error (nM^2)

        # add row to table -
        push!(rows, (
            TF_pM = round(tf_values[idx] * 1e12, digits=1),
            Actual_Peak_nM = round(actual_peak, digits=1),
            Pred_Peak_nM = round(pred_peak, digits=1),
            Actual_tpeak_s = round(actual_tpeak, digits=0),
            Pred_tpeak_s = round(pred_tpeak, digits=0),
            MSE_nM2 = round(mse, digits=2)
        ));
    end

    # display as a formatted table -
    df = DataFrame(rows);
    # pretty_table(df, tf=tf_markdown)
end

UndefVarError: UndefVarError: `X_norm` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

___

## Summary
In this example, we generated thrombin generation time series data from the Hockin-Mann coagulation model, trained an Elman RNN to predict full thrombin curves conditioned on tissue factor concentration, and evaluated predictions using autoregressive rollout.

> __Key Takeaways:__
>
> * **ODE models provide structured training data for RNNs:** The Hockin-Mann coagulation model generates thrombin curves that vary systematically with tissue factor concentration, providing a controlled dataset for training and evaluating sequence prediction models.
> * **Teacher forcing stabilizes training but autoregressive rollout tests generalization:** During training, providing the true previous value at each step prevents error accumulation. During evaluation, the model must rely on its own predictions, which reveals how well it has learned the underlying dynamics.
> * **Elman RNNs capture short-range temporal dependencies in time series:** The RNN learns to predict thrombin generation curves at unseen TF concentrations by maintaining a hidden state that encodes information about the curve trajectory and the conditioning parameter.

For more on RNN architectures and training challenges, see the [L12a lecture on Recurrent Neural Networks](CHEME-5820-L12a-Lecture-RecurrentNetworks-Spring-2026.ipynb).
___